In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
label_counts = {
    #"research_type": 0,
    # "result_outcome": 0,
    #"affiliation": 0,
    # "problem_description": 0,
    # "goal_objective": 0,
    # "research_method": 0,
    # "research_question": 0,
    # "hypothesis": 0,
    # "prediction": 0,
    # "contribution": 0,
    "pseudocode": 0,
    "open_source_code": 0,
    # "open_experiment_code": 0,
    "train": 0,
    "validation": 0,
    # "test": 0,
    # "results": 0,
    "hardware_specification": 0,
    "software_dependencies": 0,
    "experiment_setup": 0,
}

paper_counts = {
    "AAAI": {
        2014: 0,
        2015: 0,
        2016: 0,
        2017: 0,
        2018: 0,
        2019: 0,
        2020: 0,
        2021: 0,
        2022: 0,
        2023: 0,
        2024: 0,
    },
    "ICLR": {
        2014: 0,
        2015: 0,
        2016: 0,
        2017: 0,
        2018: 0,
        2019: 0,
        2020: 0,
        2021: 0,
        2022: 0,
        2023: 0,
        2024: 0,
    },
    "ICML": {
        2014: 0,
        2015: 0,
        2016: 0,
        2017: 0,
        2018: 0,
        2019: 0,
        2020: 0,
        2021: 0,
        2022: 0,
        2023: 0,
        2024: 0,
    },
    "IJCAI": {
        2014: 0,
        2015: 0,
        2016: 0,
        2017: 0,
        2018: 0,
        2019: 0,
        2020: 0,
        2021: 0,
        2022: 0,
        2023: 0,
        2024: 0,
    },
    "NeurIPS": {
        2014: 0,
        2015: 0,
        2016: 0,
        2017: 0,
        2018: 0,
        2019: 0,
        2020: 0,
        2021: 0,
        2022: 0,
        2023: 0,
        2024: 0,
    },
    "All Papers": {
        2014: 0,
        2015: 0,
        2016: 0,
        2017: 0,
        2018: 0,
        2019: 0,
        2020: 0,
        2021: 0,
        2022: 0,
        2023: 0,
        2024: 0,
    },
}

In [ ]:
checklists_introduced = {
    "AAAI": [2021],
    "ICLR": [2022],
    "ICML": [2023],
    "IJCAI": [2021],
    "NeurIPS": [2019],
    "All Papers": [2019],
}

In [ ]:
label_results = {}

label_results["All Papers"] = {
    2014: label_counts.copy(),
    2015: label_counts.copy(),
    2016: label_counts.copy(),
    2017: label_counts.copy(),
    2018: label_counts.copy(),
    2019: label_counts.copy(),
    2020: label_counts.copy(),
    2021: label_counts.copy(),
    2022: label_counts.copy(),
    2023: label_counts.copy(),
    2024: label_counts.copy(),
}

In [ ]:
exclude_theoretical = True

# Open llm_results.csv
csv_file_path = "../../results/llm_results.csv"

if not os.path.exists(csv_file_path):
    raise FileNotFoundError(f"CSV file not found: {csv_file_path}")

df = pd.read_csv(csv_file_path)

for _, row in df.iterrows():

    year = row["year"]
    conference = row["conference"]
    
    if conference not in label_results:
        label_results[conference] = {
            2014: label_counts.copy(),
            2015: label_counts.copy(),
            2016: label_counts.copy(),
            2017: label_counts.copy(),
            2018: label_counts.copy(),
            2019: label_counts.copy(),
            2020: label_counts.copy(),
            2021: label_counts.copy(),
            2022: label_counts.copy(),
            2023: label_counts.copy(),
            2024: label_counts.copy(),
        }
    
    # If we are excluding theoretical papers skip papers that are theoretical
    if exclude_theoretical and int(row["research_type"]) == 1:
        continue

    for label in label_counts.keys():
        # Normalize run label: treat 1 and 2 as 1 (positive)
        if row[label] == 1 or row[label] == 2:
            label_results[conference][year][label] += 1
            label_results["All Papers"][year][label] += 1

    paper_counts[conference][year] += 1
    paper_counts["All Papers"][year] += 1

    
# Convert label_results into perecentage format
for conference, years in label_results.items():
    for year, labels in years.items():
        total_papers = paper_counts[conference][year]
        if total_papers > 0:
            for label, count in labels.items():
                labels[label] = round((count / total_papers) * 100, 2)
        else:
            for label in labels.keys():
                labels[label] = 0.0    

# Remove 2014 from IJCAI
label_results["IJCAI"].pop(2014, None)

In [ ]:
for conference in label_results.keys():

    # Create a DataFrame from the label results
    df = pd.DataFrame(label_results[conference])

    print(f"\nConference: {conference}")
    # Display the DataFrame
    print(df)

In [ ]:
evaluations_file = "../../evaluations.json"

sota_evaluations = {
    "AAAI": {
        2014: label_counts.copy(),
        2016: label_counts.copy(),
    },
    "IJCAI": {
        2016: label_counts.copy(),
    },
}

AAAI_14_count = 0
IJCAI_16_count = 0
AAAI_16_count = 0

# read the evaluations file
if os.path.exists(evaluations_file):
    with open(evaluations_file, 'r') as file:
        evaluations = json.load(file)

    for evaluation in evaluations:

        # If we are excluding theoretical papers skip papers that are theoretical
        if exclude_theoretical and int(evaluation["research_type"]) == 1:
            continue

        if evaluation["conference"] == "AAAI 14":
            #print("AAAI 14")
            for label in label_counts.keys():
                if evaluation[label] == 1 or evaluation[label] == 2:
                    sota_evaluations["AAAI"][2014][label] += 1
            AAAI_14_count += 1
        elif evaluation["conference"] == "IJCAI 16":
            #print("IJCAI 16")
            for label in label_counts.keys():
                if evaluation[label] == 1 or evaluation[label] == 2:
                    sota_evaluations["IJCAI"][2016][label] += 1
            IJCAI_16_count += 1
        elif evaluation["conference"] == "AAAI 16":
            #print("AAAI 16")
            for label in label_counts.keys():
                if evaluation[label] == 1 or evaluation[label] == 2:
                    sota_evaluations["AAAI"][2016][label] += 1
            AAAI_16_count += 1
        else:
            continue

# Convert sota_evaluations into percentage format
for conference, years in sota_evaluations.items():
    for year, labels in years.items():
        if conference == "AAAI" and year == 2014:
            total_papers = AAAI_14_count
        elif conference == "IJCAI" and year == 2016:
            total_papers = IJCAI_16_count
        elif conference == "AAAI" and year == 2016:
            total_papers = AAAI_16_count

        if total_papers > 0:
            for label, count in labels.items():
                labels[label] = round((count / total_papers) * 100, 2)
        else:
            for label in labels.keys():
                labels[label] = 0.0

print(sota_evaluations)

In [ ]:
for conference in sota_evaluations.keys():

    # Create a DataFrame from the label results
    df = pd.DataFrame(sota_evaluations[conference])

    print(f"\nConference: {conference}")
    # Display the DataFrame
    print(df)

In [ ]:
name_mapping = {
    "research_type": "Research Type",
    # "result_outcome": "Result Outcome",
    "affiliation": "Affiliation",
    # "problem_description": "Problem Description",
    # "goal_objective": "Goal/Objective",
    # "research_method": "Research Method",
    # "research_question": "Research Question",
    # "hypothesis": "Hypothesis",
    # "prediction": "Prediction",
    # "contribution": "Contribution",
    "pseudocode": "Pseudocode",
    "open_source_code": "Open Code",
    # "open_experiment_code": "Open Experiment Code",
    "train": "Open Datasets",
    "validation": "Dataset Splits",
    # "test": "Test",
    # "results": "Results",
    "hardware_specification": "Hardware Specification",
    "software_dependencies": "Software Dependencies",
    "experiment_setup": "Experiment Setup",
}

In [ ]:
# Determine the number of rows needed for two columns
num_conferences = len(label_results.keys())
num_rows = (num_conferences + 1) // 2  # Add 1 to handle odd number of conferences
num_columns = 2

figure_height = 6 * num_rows
figure_width = 16

font_size = 18
line_width = 2

annotate_points = False
show_regular_lines = True
show_trendlines = False
show_split_trendlines = True
show_checkslists = True
show_sota_evaluations = False
show_paper_counts = False
only_all_papers = False

if only_all_papers:
    # Remove Conferences for label_results
    label_results.pop("AAAI", None)
    label_results.pop("ICLR", None)
    label_results.pop("ICML", None)
    label_results.pop("IJCAI", None)
    label_results.pop("NeurIPS", None)

    num_conferences = 1
    num_rows = 1
    num_columns = 1
    font_size = 20
    figure_height = 8
    line_width = 3

fig, axes = plt.subplots(num_rows, num_columns, figsize=(figure_width, figure_height), constrained_layout=True)

if isinstance(axes, np.ndarray):
    axes = axes.flatten()
else:
    axes = [axes]

legend_lines = {}

for idx, conference in enumerate(label_results.keys()):
    ax = axes[idx]
    years = list(label_results[conference].keys())
    if show_paper_counts:
        year_labels = [f"{year}\n({paper_counts[conference][year]})" for year in years]
    else:
        year_labels = [f"{year}" for year in years]
    label_lines = {}

    for label in label_results[conference][years[0]].keys():
        counts = [label_results[conference][year][label] for year in years]
        display_label = name_mapping.get(label, label)
        line_label = display_label if show_regular_lines else "_nolegend_"
        trendline_label = "_nolegend_" if show_regular_lines else display_label
        line, = ax.plot(year_labels, counts, label=line_label, visible=show_regular_lines)
        label_lines[label] = line

        line.set_linewidth(line_width)
        legend_lines.setdefault(display_label, line)

        # Annotate each point
        if annotate_points:
            for x, y in zip(year_labels, counts):
                ax.text(x, y, str(f"{y}%"), fontsize=9, ha='center', va='bottom')

        # Fit and plot a linear trendline with the same color
        if show_trendlines:
            z = np.polyfit(range(len(years)), counts, 1)
            p = np.poly1d(z)
            ax.plot(year_labels, p(range(len(years))), linestyle='--', alpha=0.5, linewidth=line_width, color=line.get_color(), label=trendline_label)
            trendline_label = "_nolegend_"

        # Split trendlines at checklist introduction
        if show_split_trendlines:
            checklist_years = checklists_introduced.get(conference, [])
            if checklist_years:
                split_idx = years.index(checklist_years[0]) if checklist_years[0] in years else None
                if split_idx is not None and split_idx > 0:
                    # Before checklist
                    x1 = np.arange(0, split_idx + 1)
                    y1 = counts[:split_idx + 1]
                    if len(x1) > 1:
                        z1 = np.polyfit(x1, y1, 1)
                        p1 = np.poly1d(z1)
                        ax.plot([year_labels[i] for i in x1], p1(x1), linestyle='--', alpha=0.7, linewidth=line_width, color=line.get_color(), label=trendline_label)
                        trendline_label = "_nolegend_"
                    # After checklist
                    x2 = np.arange(split_idx, len(years))
                    y2 = counts[split_idx:]
                    if len(x2) > 1:
                        z2 = np.polyfit(x2, y2, 1)
                        p2 = np.poly1d(z2)
                        ax.plot([year_labels[i] for i in x2], p2(x2), linestyle='--', alpha=0.7, linewidth=line_width, color=line.get_color())
                else:
                    # If checklist year not in years, fallback to full trendline
                    z = np.polyfit(range(len(years)), counts, 1)
                    p = np.poly1d(z)
                    ax.plot(year_labels, p(range(len(years))), linestyle='--', alpha=0.5, linewidth=line_width, color=line.get_color(), label=trendline_label)
                    trendline_label = "_nolegend_"

    # Add sota_evaluation points for AAAI and IJCAI (no text, no legend)
    if show_sota_evaluations and conference in sota_evaluations:
        for year, eval_labels in sota_evaluations[conference].items():
            for label, value in eval_labels.items():
                if label in label_lines:
                    color = label_lines[label].get_color()
                    if year in years:
                        x_idx = years.index(year)
                        ax.scatter(year_labels[x_idx], value, color=color, marker='o', s=80, alpha=0.5, edgecolor=color, linewidths=line_width, zorder=5)

    # Add vertical lines for checklist introduction years
    if show_checkslists:
        for checklist_year in checklists_introduced.get(conference, []):
            if checklist_year in years:
                idx = years.index(checklist_year)
                if conference == "All Papers":
                    ax.axvline(x=year_labels[idx], color='red', linestyle=':', linewidth=line_width, label=f'First Checklist Introduced')
                else:
                    ax.axvline(x=year_labels[idx], color='red', linestyle=':', linewidth=line_width, label=f'Checklist Introduced')

    if show_paper_counts:
        ax.set_xlabel("Year (Paper Count)", fontsize=font_size - 2)
    else:
        ax.set_xlabel("Year", fontsize=font_size - 2)
    if exclude_theoretical:
        ax.set_ylabel("Percentage of Empirical Papers", fontsize=font_size - 2)
    else:
        ax.set_ylabel("Percentage of All Papers", fontsize=font_size - 2)
    ax.set_title(f"{conference} Results with Trendlines", fontsize=font_size)
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_locator(mticker.MultipleLocator(10))  # Set y grid every 10
    ax.grid(True, axis='y')

    # Show every other year on x-axis for readability
    for i, label in enumerate(ax.xaxis.get_ticklabels()):
        if i % 2 == 0:
            label.set_visible(True)
        else:
            label.set_visible(False)

    # Show y-axis ticks and labels on both left and right
    ax.yaxis.set_ticks_position('both')
    ax.tick_params(axis='y', labelright=True)

    # Set the font size for tick labels
    ax.tick_params(axis='both', which='major', labelsize=font_size - 4)


# Remove unused subplots
for i in range(idx + 1, len(axes)):
    fig.delaxes(axes[i])

# Add a single legend for the entire figure. Use solid, fully opaque proxies
# so the legend remains readable when only trendlines are shown.
legend_handles = [
    type(line)([], [], color=line.get_color(), linestyle='-', linewidth=line_width, alpha=1, label=label)
    for label, line in legend_lines.items()
]
legend_labels = list(legend_lines.keys())
fig.legend(legend_handles, legend_labels, loc='upper center', bbox_to_anchor=(.5, -.01), ncol=4, fontsize=font_size - 4)

# Save the figure
if not os.path.exists("../../plots/trendlines"):
    os.makedirs("../../plots/trendlines")

plt.savefig(f"../../plots/trendlines/all_conferences.pdf", dpi=600, bbox_inches='tight')
plt.show()


In [ ]:
# Prepare a table of slopes for all conferences, before and after checklist introduction
slope_data = []

for conf in label_results.keys():
    years = list(label_results[conf].keys())
    checklist_years = checklists_introduced.get(conf, [])
    if not checklist_years:
        continue
    checklist_year = checklist_years[0]
    if checklist_year not in years:
        continue
    split_idx = years.index(checklist_year)
    for label in label_counts.keys():
        counts = [label_results[conf][year][label] for year in years]
        # Before checklist (exclude checklist year)
        x1 = np.arange(0, split_idx)
        y1 = counts[:split_idx]
        years_before = years[:split_idx]
        slope_before = np.polyfit(x1, y1, 1)[0] if len(x1) > 1 else np.nan
        # After checklist (include checklist year)
        x2 = np.arange(split_idx, len(years))
        y2 = counts[split_idx:]
        years_after = years[split_idx:]
        slope_after = np.polyfit(x2, y2, 1)[0] if len(x2) > 1 else np.nan
        slope_data.append({
            "conference": conf,
            "label": label,
            "slope_before": round(slope_before, 2) if not np.isnan(slope_before) else np.nan,
            "years_before": years_before,
            "slope_after": round(slope_after, 2) if not np.isnan(slope_after) else np.nan,
            "years_after": years_after
        })

slope_df = pd.DataFrame(slope_data)


In [ ]:
slope_df.style.hide(axis="index")

In [ ]:
annotate_points = False
show_trendlines = False
include_all_papers = True

conferences = list(label_results.keys())
labels = list(label_results[conferences[0]][list(label_results[conferences[0]].keys())[0]].keys())

for label in labels:
    plt.figure(figsize=(12, 6))
    conference_lines = {}

    for conference in conferences:
        # Skip "All Papers" if not including all papers
        if not include_all_papers:
            if conference == "All Papers":
                continue
        years = list(label_results[conference].keys())
        year_labels = [f"{year}" for year in years]
        counts = [label_results[conference][year][label] for year in years]
        line, = plt.plot(year_labels, counts, label=conference)
        conference_lines[conference] = line

        line.set_linewidth(2)
        # make all papers dashed
        if conference == "All Papers":
            # Set the color to black and dashed and increase the thickness
            line.set_color('black')
            line.set_linestyle('--')
            line.set_linewidth(2)

        # Annotate each point
        if annotate_points:
            for x, y in zip(year_labels, counts):
                plt.text(x, y, str(f"{y}%"), fontsize=9, ha='center', va='bottom')

        # Fit and plot a linear trendline with the same color
        if show_trendlines:
            z = np.polyfit(range(len(years)), counts, 1)
            p = np.poly1d(z)
            plt.plot(year_labels, p(range(len(years))), linestyle='--', alpha=0.5, line_width=2, color=line.get_color())

    # Add sota_evaluation points for each conference if enabled
    if show_sota_evaluations:
        for conference in conferences:
            if conference == "All Papers":
                continue
            if conference in sota_evaluations:
                years = list(label_results[conference].keys())
                year_labels = [f"{year}" for year in years]
                line = conference_lines.get(conference)
                if line is None:
                    continue
                color = line.get_color()
                for year, eval_labels in sota_evaluations[conference].items():
                    if label in eval_labels and year in years:
                        x_idx = years.index(year)
                        plt.scatter(year_labels[x_idx], eval_labels[label], color=color, marker='o', s=80, alpha=0.5, edgecolor=color, linewidths=2, zorder=5)



    plt.xlabel("Year", fontsize=font_size - 2)
    if exclude_theoretical:
        plt.ylabel("Percentage of Empirical Papers", fontsize=font_size - 2)
    else:
        plt.ylabel("Percentage of All Papers", fontsize=font_size - 2)
    display_label = name_mapping.get(label, label)
    plt.title(f'{display_label} Results for Each Conference', fontsize=font_size)
    plt.ylim(0, 100)

    # Move the legend to under the plot
    plt.legend(bbox_to_anchor=(0.5, -0.15), loc='upper center', borderaxespad=0., ncol=6, fontsize=font_size - 4)

    plt.gca().yaxis.set_major_locator(mticker.MultipleLocator(10))  # Set y grid every 10
    plt.grid(True, axis='y')

    # Show y-axis ticks and labels on both left and right
    plt.gca().yaxis.set_ticks_position('both')
    plt.gca().tick_params(axis='y', labelright=True)

    # Increase font size of tick labels
    plt.xticks(fontsize=font_size - 4)
    plt.yticks(fontsize=font_size - 4)

    if not os.path.exists("../../plots/trendlines/labels"):
        os.makedirs("../../plots/trendlines/labels")
 
    plt.savefig(f"../../plots/trendlines/labels/{display_label}.pdf", dpi=600, bbox_inches='tight')

    plt.show()

In [ ]:
annotate_points = False
show_trendlines = False
include_all_papers = True

conferences = list(label_results.keys())
labels = list(label_results[conferences[0]][list(label_results[conferences[0]].keys())[0]].keys())

# Create subplots: 2 columns, enough rows to fit all labels
ncols = 2
nrows = int(np.ceil(len(labels) / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(18, 6 * nrows))
axes = axes.flatten()

# Store legend handles/labels from the first subplot
legend_handles = []
legend_labels = []

for idx, label in enumerate(labels):
    ax = axes[idx]
    conference_lines = {}

    for conference in conferences:
        # Skip "All Papers" if not including all papers
        if not include_all_papers:
            if conference == "All Papers":
                continue
        years = list(label_results[conference].keys())
        year_labels = [f"{year}" for year in years]
        counts = [label_results[conference][year][label] for year in years]
        line, = ax.plot(year_labels, counts, label=conference)
        conference_lines[conference] = line

        # make all papers dashed
        if conference == "All Papers":
            # Set the color to black and dashed and increase the thickness
            line.set_color('black')
            line.set_linestyle('--')
            line.set_linewidth(line_width)

        if idx == 0:  # Only collect legend info once
            legend_handles.append(line)
            legend_labels.append(conference)

        if annotate_points:
            for x, y in zip(year_labels, counts):
                ax.text(x, y, str(f"{y}%"), fontsize=9, ha='center', va='bottom')

        if show_trendlines:
            z = np.polyfit(range(len(years)), counts, 1)
            p = np.poly1d(z)
            ax.plot(year_labels, p(range(len(years))), linestyle='--', alpha=0.5, color=line.get_color())

    if show_sota_evaluations:
        for conference in conferences:
            if conference == "All Papers":
                continue
            if conference in sota_evaluations:
                years = list(label_results[conference].keys())
                year_labels = [f"{year}" for year in years]
                line = conference_lines.get(conference)
                if line is None:
                    continue
                color = line.get_color()
                for year, eval_labels in sota_evaluations[conference].items():
                    if label in eval_labels and year in years:
                        x_idx = years.index(year)
                        ax.scatter(year_labels[x_idx], eval_labels[label],
                                   color=color, marker='o', s=80, alpha=0.5,
                                   edgecolor=color, linewidths=2, zorder=5)

    ax.set_xlabel("Year", fontsize=font_size - 2)
    if exclude_theoretical:
        ax.set_ylabel("Percentage of Empirical Papers", fontsize=font_size - 2)
    else:
        ax.set_ylabel("Percentage of All Papers", fontsize=font_size - 2)
    display_label = name_mapping.get(label, label)
    ax.set_title(f'{display_label} Results by Conference', fontsize=font_size)
    ax.set_ylim(0, 100)
    ax.yaxis.set_major_locator(mticker.MultipleLocator(10))
    ax.grid(True, axis='y')
    ax.yaxis.set_ticks_position('both')
    ax.tick_params(axis='y', labelright=True)

    # Set the font size for tick labels
    ax.tick_params(axis='both', which='major', labelsize=font_size - 4)

    # Show every other year on x-axis for readability
    for i, label in enumerate(ax.xaxis.get_ticklabels()):
        if i % 2 == 0:
            label.set_visible(True)
        else:
            label.set_visible(False)

# Hide unused subplots
for j in range(len(labels), len(axes)):
    fig.delaxes(axes[j])

# Adjust subplot spacing so bottom margin is tight
fig.subplots_adjust(hspace=0.2, bottom=0.04)  # bottom controls gap between last plot and legend

# Add one legend for the entire figure, close to the last plots
fig.legend(
    legend_handles, legend_labels,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.0),  # smaller positive y keeps it close
    ncol=6,
    fontsize=font_size - 2
)

plt.savefig("../../plots/trendlines/labels/all_labels.pdf", dpi=600, bbox_inches='tight')
plt.show()